🛠️ Requisitos previos
La base de datos `dw_universidad` se crea automáticamente en la celda de conexión (si no existe).

> ⚠️ Si aparece `UnicodeDecodeError: 'utf-8' codec can't decode byte 0xab`, significa que la base **no existe** y psycopg2 no puede decodificar el mensaje de error en español de PostgreSQL (`«...»` viene en cp1252). La celda siguiente ya lo resuelve creando la base.

Lectura desde MongoDB Silver y conexión a PostgreSQL

In [9]:
# Dependencias del proyecto: en la raíz ejecutar ->  pip install -r requirements.txt

In [10]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sqlalchemy import create_engine, text

BASE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(BASE_DIR))

from src.conn import get_mongo_db, get_pg_engine

db = get_mongo_db()
col_silver = db["estudiantes_limpios"]

# --- Crear la base si no existe ---
# Conectarse directo a 'dw_universidad' cuando no existe dispara un
# UnicodeDecodeError: psycopg2 no puede decodificar como UTF-8 el mensaje
# en español de PostgreSQL ('la base de datos «...» no existe').
pg_engine = get_pg_engine()
url = pg_engine.url
admin_engine = create_engine(url.set(database="postgres"), isolation_level="AUTOCOMMIT")
with admin_engine.connect() as conn_admin:
    existe = conn_admin.execute(
        text("SELECT 1 FROM pg_database WHERE datname = :nombre"),
        {"nombre": url.database},
    ).scalar()
    if not existe:
        conn_admin.execute(text(f'CREATE DATABASE "{url.database}"'))
        print(f"Base de datos '{url.database}' creada")
admin_engine.dispose()

df = pd.DataFrame(list(col_silver.find({}, {"_id": 0})))
print(f"Documentos leídos desde MongoDB Silver: {len(df):,} (base: {db.name})")
print(f"Data Warehouse destino: {url.database}")
display(df.head(3))


Documentos leídos desde MongoDB Silver: 5,000 (base: Estudiantes)
Data Warehouse destino: dw_universidad


,id_alumno,edad_alumno,carrera,cohorte,codigo_materia,nombre_materia,anio_cursada,periodo_inscripcion,tipo_instancia,fecha_examen,nota,asistencias,clases_totales,pct_asistencia
0,A0164,23.0,Ingeniería en Sistemas,2018,ALG-01,Algoritmos,1,2024-1,Parcial 2,2023-11-06,9.0,8.0,20,40.0
1,A0056,31.0,Ingeniería en Sistemas,2020,PL-01,Programación Lógica,1,2023-1,TP Práctico,2024-09-25,8.0,16.0,40,40.0
2,A0041,21.0,Ingeniería en Sistemas,2023,FIS-01,Física,1,2024-1,Final,2023-08-07,7.0,13.0,20,65.0


Creación del Esquema en Estrella (DDL en PostgreSQL)

In [11]:
sql_ddl = """
CREATE SCHEMA IF NOT EXISTS dw_academico;
SET search_path TO dw_academico;

CREATE TABLE IF NOT EXISTS dim_carrera (
    sk_carrera SERIAL PRIMARY KEY,
    nombre_carrera VARCHAR(120) UNIQUE NOT NULL
);

CREATE TABLE IF NOT EXISTS dim_estudiante (
    sk_estudiante SERIAL PRIMARY KEY,
    id_alumno VARCHAR(20) UNIQUE NOT NULL,
    edad_actual INT CHECK (edad_actual BETWEEN 16 AND 100),
    cohorte_ingreso INT NOT NULL,
    sk_carrera INT REFERENCES dim_carrera(sk_carrera)
);

CREATE TABLE IF NOT EXISTS dim_materia (
    sk_materia SERIAL PRIMARY KEY,
    codigo_materia VARCHAR(15) UNIQUE NOT NULL,
    nombre_materia VARCHAR(120) NOT NULL,
    anio_cursada INT NOT NULL CHECK (anio_cursada BETWEEN 1 AND 6)
);

CREATE TABLE IF NOT EXISTS dim_periodo (
    sk_periodo SERIAL PRIMARY KEY,
    periodo_inscripcion VARCHAR(15) UNIQUE NOT NULL,
    anio_lectivo INT NOT NULL,
    cuatrimestre INT NOT NULL CHECK (cuatrimestre IN (1, 2))
);

CREATE TABLE IF NOT EXISTS dim_instancia (
    sk_instancia SERIAL PRIMARY KEY,
    tipo_instancia VARCHAR(50) UNIQUE NOT NULL
);

CREATE TABLE IF NOT EXISTS dim_docente (
    sk_docente SERIAL PRIMARY KEY,
    legajo_docente VARCHAR(20) UNIQUE NOT NULL,
    nombre_docente VARCHAR(100) NOT NULL
);

INSERT INTO dim_docente (legajo_docente, nombre_docente) 
VALUES ('DOC-000', 'Cátedra General')
ON CONFLICT (legajo_docente) DO NOTHING;

CREATE TABLE IF NOT EXISTS fact_rendimiento_academico (
    id_hecho BIGSERIAL PRIMARY KEY,
    sk_estudiante INT NOT NULL REFERENCES dim_estudiante(sk_estudiante),
    sk_materia INT NOT NULL REFERENCES dim_materia(sk_materia),
    sk_carrera INT NOT NULL REFERENCES dim_carrera(sk_carrera),
    sk_periodo INT NOT NULL REFERENCES dim_periodo(sk_periodo),
    sk_instancia INT NOT NULL REFERENCES dim_instancia(sk_instancia),
    sk_docente INT NOT NULL REFERENCES dim_docente(sk_docente),
    fecha_examen DATE NOT NULL,
    nota NUMERIC(4,2) CHECK (nota BETWEEN 0 AND 10),
    asistencias INT CHECK (asistencias >= 0),
    clases_totales INT NOT NULL CHECK (clases_totales > 0),
    pct_asistencia NUMERIC(5,2) CHECK (pct_asistencia BETWEEN 0 AND 100),
    estado_aprobacion VARCHAR(20) NOT NULL,
    es_aprobado BOOLEAN NOT NULL
);

CREATE INDEX IF NOT EXISTS idx_fact_estudiante ON fact_rendimiento_academico(sk_estudiante);
CREATE INDEX IF NOT EXISTS idx_fact_materia ON fact_rendimiento_academico(sk_materia);
CREATE INDEX IF NOT EXISTS idx_fact_periodo ON fact_rendimiento_academico(sk_periodo);
CREATE INDEX IF NOT EXISTS idx_fact_fecha ON fact_rendimiento_academico(fecha_examen);
"""

with pg_engine.begin() as conn:
    conn.execute(text(sql_ddl))
print("✅ Esquema en estrella creado en PostgreSQL (schema: dw_academico).")

✅ Esquema en estrella creado en PostgreSQL (schema: dw_academico).


Transformación Gold y Carga al Data Warehouse

In [12]:
from src.gold import construir_dimensiones, construir_hechos, preparar_hechos

# Reglas de negocio (ver src/gold.py y sus tests):
#  - edad consolidada por alumno (mediana)
#  - asistencia imputada SOLO donde falta (no se sobreescribe el dato original)
#  - Aprobado si nota >= 6; Desaprobado si nota < 6; Ausente si no hay nota
#  - la carrera del hecho se deriva del ALUMNO (carrera canónica)
df = preparar_hechos(df)
dims = construir_dimensiones(df)

with pg_engine.begin() as conn:
    conn.execute(text("SET search_path TO dw_academico;"))
    conn.execute(text(
        "TRUNCATE fact_rendimiento_academico, dim_estudiante, dim_materia, "
        "dim_carrera, dim_periodo, dim_instancia, dim_docente RESTART IDENTITY CASCADE;"
    ))

    dims["dim_carrera"].to_sql("dim_carrera", conn, schema="dw_academico", if_exists="append", index=False)
    map_carrera = pd.read_sql("SELECT nombre_carrera, sk_carrera FROM dw_academico.dim_carrera", conn).set_index("nombre_carrera")["sk_carrera"].to_dict()

    est = dims["dim_estudiante"].copy()
    est["sk_carrera"] = est["nombre_carrera"].map(map_carrera)
    est[["id_alumno", "edad_actual", "cohorte_ingreso", "sk_carrera"]].to_sql(
        "dim_estudiante", conn, schema="dw_academico", if_exists="append", index=False
    )
    map_estudiante = pd.read_sql("SELECT id_alumno, sk_estudiante FROM dw_academico.dim_estudiante", conn).set_index("id_alumno")["sk_estudiante"].to_dict()

    dims["dim_materia"].to_sql("dim_materia", conn, schema="dw_academico", if_exists="append", index=False)
    map_materia = pd.read_sql("SELECT codigo_materia, sk_materia FROM dw_academico.dim_materia", conn).set_index("codigo_materia")["sk_materia"].to_dict()

    dims["dim_periodo"].to_sql("dim_periodo", conn, schema="dw_academico", if_exists="append", index=False)
    map_periodo = pd.read_sql("SELECT periodo_inscripcion, sk_periodo FROM dw_academico.dim_periodo", conn).set_index("periodo_inscripcion")["sk_periodo"].to_dict()

    dims["dim_instancia"].to_sql("dim_instancia", conn, schema="dw_academico", if_exists="append", index=False)
    map_instancia = pd.read_sql("SELECT tipo_instancia, sk_instancia FROM dw_academico.dim_instancia", conn).set_index("tipo_instancia")["sk_instancia"].to_dict()

    dims["dim_docente"].to_sql("dim_docente", conn, schema="dw_academico", if_exists="append", index=False)
    map_docente = pd.read_sql("SELECT legajo_docente, sk_docente FROM dw_academico.dim_docente", conn).set_index("legajo_docente")["sk_docente"].to_dict()

    hechos = construir_hechos(df, {
        "carrera": map_carrera, "estudiante": map_estudiante, "materia": map_materia,
        "periodo": map_periodo, "instancia": map_instancia, "docente": map_docente,
    })
    hechos.to_sql("fact_rendimiento_academico", conn, schema="dw_academico", if_exists="append", index=False, chunksize=1000)

print(f"🥇 Capa Gold cargada exitosamente: {len(hechos):,} hechos en PostgreSQL.")
print(f"   Docentes en dim_docente: {len(dims['dim_docente'])} | Carreras: {len(dims['dim_carrera'])}")


🥇 Capa Gold cargada exitosamente: 5,000 hechos en PostgreSQL.
   Docentes en dim_docente: 1 | Carreras: 1


In [13]:
# ---- Validación automática del modelo dimensional ----
with pg_engine.connect() as conn:
    fk = conn.execute(text("""
        SELECT count(*) FROM information_schema.table_constraints
        WHERE constraint_schema = 'dw_academico' AND constraint_type = 'FOREIGN KEY'
    """)).scalar()
    cheques = pd.read_sql(text("""
        SELECT
            (SELECT count(*) FROM dw_academico.fact_rendimiento_academico) AS hechos,
            (SELECT count(*) FROM dw_academico.dim_estudiante) AS estudiantes,
            (SELECT count(*) FROM dw_academico.fact_rendimiento_academico f
             JOIN dw_academico.dim_estudiante e USING (sk_estudiante)
             WHERE e.sk_carrera <> f.sk_carrera) AS carrera_inconsistente,
            (SELECT count(*) FROM dw_academico.fact_rendimiento_academico f
             LEFT JOIN dw_academico.dim_materia m ON f.sk_materia = m.sk_materia
             LEFT JOIN dw_academico.dim_periodo p ON f.sk_periodo = p.sk_periodo
             LEFT JOIN dw_academico.dim_instancia i ON f.sk_instancia = i.sk_instancia
             LEFT JOIN dw_academico.dim_docente d ON f.sk_docente = d.sk_docente
             WHERE m.sk_materia IS NULL OR p.sk_periodo IS NULL
                OR i.sk_instancia IS NULL OR d.sk_docente IS NULL) AS hechos_huerfanos
    """), conn)

print(f"Claves foráneas en el esquema: {fk}")
display(cheques)

assert cheques["carrera_inconsistente"].iloc[0] == 0, "el hecho no coincide con la carrera del alumno"
assert cheques["hechos_huerfanos"].iloc[0] == 0, "hay hechos sin dimensión válida"
print("✅ Integridad dimensional verificada.")


Claves foráneas en el esquema: 7


,hechos,estudiantes,carrera_inconsistente,hechos_huerfanos
0,5000,200,0,0


✅ Integridad dimensional verificada.


Creación de Vistas Analíticas para Análisis y Power BI / Tableau

In [14]:
sql_views = """
SET search_path TO dw_academico;

-- Vista 1: Rendimiento por Materia
CREATE OR REPLACE VIEW vw_rendimiento_por_materia AS
SELECT 
    m.codigo_materia,
    m.nombre_materia,
    m.anio_cursada,
    COUNT(f.id_hecho) AS total_evaluaciones,
    ROUND(AVG(f.nota), 2) AS promedio_nota,
    ROUND(AVG(f.pct_asistencia), 1) AS asistencia_media_pct,
    ROUND(100.0 * SUM(CASE WHEN NOT f.es_aprobado THEN 1 ELSE 0 END) / COUNT(f.id_hecho), 1) AS pct_no_aprobacion,
    ROUND(100.0 * SUM(CASE WHEN f.estado_aprobacion = 'Desaprobado' THEN 1 ELSE 0 END) / COUNT(f.id_hecho), 1) AS pct_desaprobados_reales,
    ROUND(100.0 * SUM(CASE WHEN f.estado_aprobacion = 'Ausente' THEN 1 ELSE 0 END) / COUNT(f.id_hecho), 1) AS pct_ausentismo
FROM fact_rendimiento_academico f
JOIN dim_materia m ON f.sk_materia = m.sk_materia
GROUP BY m.codigo_materia, m.nombre_materia, m.anio_cursada;

-- Vista 2: Evolución por Período, Carrera y Año de Cursada
CREATE OR REPLACE VIEW vw_evolucion_periodo_carrera AS
SELECT 
    p.periodo_inscripcion,
    p.anio_lectivo,
    p.cuatrimestre,
    c.nombre_carrera,
    m.anio_cursada,
    COUNT(f.id_hecho) AS evaluaciones,
    ROUND(AVG(f.nota), 2) AS promedio_cuatrimestral,
    ROUND(AVG(f.pct_asistencia), 1) AS asistencia_promedio,
    ROUND(100.0 * SUM(CASE WHEN NOT f.es_aprobado THEN 1 ELSE 0 END) / COUNT(f.id_hecho), 1) AS tasa_desaprobacion
FROM fact_rendimiento_academico f
JOIN dim_periodo p ON f.sk_periodo = p.sk_periodo
JOIN dim_carrera c ON f.sk_carrera = c.sk_carrera
JOIN dim_materia m ON f.sk_materia = m.sk_materia
GROUP BY p.periodo_inscripcion, p.anio_lectivo, p.cuatrimestre, c.nombre_carrera, m.anio_cursada;

-- Vista 3: Alerta Temprana de Deserción por Alumno (Semáforo)
CREATE OR REPLACE VIEW vw_alerta_temprana_alumnos AS
WITH metricas_alumno AS (
    SELECT 
        e.id_alumno,
        e.edad_actual,
        e.cohorte_ingreso,
        c.nombre_carrera,
        COUNT(f.id_hecho) AS instancias_rendidas,
        COUNT(DISTINCT CASE WHEN f.es_aprobado THEN f.sk_materia END) AS materias_con_aprobacion,
        ROUND(AVG(f.nota), 2) AS promedio_general,
        ROUND(AVG(f.pct_asistencia), 1) AS asistencia_promedio,
        ROUND(100.0 * SUM(CASE WHEN NOT f.es_aprobado THEN 1 ELSE 0 END) / COUNT(f.id_hecho), 1) AS tasa_reprobacion_pct,
        SUM(CASE WHEN f.estado_aprobacion = 'Ausente' THEN 1 ELSE 0 END) AS cantidad_ausentes
    FROM fact_rendimiento_academico f
    JOIN dim_estudiante e ON f.sk_estudiante = e.sk_estudiante
    JOIN dim_carrera c ON f.sk_carrera = c.sk_carrera
    GROUP BY e.id_alumno, e.edad_actual, e.cohorte_ingreso, c.nombre_carrera
)
SELECT 
    *,
    ROUND(
        (CASE WHEN asistencia_promedio < 60 THEN 40 WHEN asistencia_promedio < 75 THEN 20 ELSE 0 END) +
        (CASE WHEN COALESCE(promedio_general, 0) < 4 THEN 35 WHEN COALESCE(promedio_general, 0) < 6 THEN 20 ELSE 0 END) +
        (CASE WHEN tasa_reprobacion_pct > 50 THEN 25 WHEN tasa_reprobacion_pct > 30 THEN 10 ELSE 0 END)
    , 0) AS score_riesgo,
    CASE 
        WHEN asistencia_promedio < 60 OR COALESCE(promedio_general, 0) < 4 OR tasa_reprobacion_pct > 60 THEN 'Riesgo Alto'
        WHEN asistencia_promedio < 75 OR COALESCE(promedio_general, 0) < 6 OR tasa_reprobacion_pct > 35 THEN 'Riesgo Medio'
        ELSE 'Riesgo Bajo'
    END AS semaforo_riesgo
FROM metricas_alumno;
"""

with pg_engine.begin() as conn:
    conn.execute(text(sql_views))
print("✅ Vistas analíticas creadas en PostgreSQL.")

✅ Vistas analíticas creadas en PostgreSQL.
